# λ_d Phase 2 — Colab Training

12-layer LDStack, D=896, K=4 on wikitext-103.

**Before run:** go to Edit → Notebook settings → Hardware accelerator → **T4 GPU**

In [ ]:
# Mount Drive for checkpoints
from google.colab import drive
drive.mount('/content/drive')

# Where to save checkpoints (change this if you want a different folder)
CKPT_DIR = '/content/drive/MyDrive/lambda_checkpoints'
import os; os.makedirs(CKPT_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {CKPT_DIR}')

In [ ]:
# Clone repo
import os, sys
REPO = '/content/EVA-Ai'
if not os.path.exists(REPO):
    !git clone https://github.com/BlackCatSpb/EVA-Ai.git {REPO}
sys.path.insert(0, REPO)
%cd {REPO}
!git pull  # get latest

In [ ]:
# Install and prepare data
!pip install -q datasets tokenizers

# Pre-generate chunks (done once, cached)
if not os.path.exists('wikitext_chunks.npy'):
    from datasets import load_dataset
    from transformers import AutoTokenizer
    ds = load_dataset('Salesforce/wikitext', 'wikitext-103-raw-v1', split='train')
    tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B')
    text = '\n\n'.join(ds['text'])
    tokens = tokenizer(text)['input_ids']
    SEQ_LEN = 128
    n = len(tokens) // SEQ_LEN
    import numpy as np
    arr = np.array(tokens[:n * SEQ_LEN], dtype=np.int32).reshape(n, SEQ_LEN)
    np.save('wikitext_chunks.npy', arr)
    print(f'Created {arr.shape[0]} chunks')
else:
    print('wikitext_chunks.npy already exists')

In [ ]:
# Run training
print(f'Using checkpoint dir: {CKPT_DIR}')
print(f'T4 VRAM: 16GB, batch_size=8 should fit')

!python colab_train.py \
    --ckpt_dir "{CKPT_DIR}" \
    --batch_size 8 \
    --n_train 50000